# Fine Tuning a VLA on Hugging Face

In [ ]:
from huggingface_hub import login
login()

### Part 0: Dataset Generation

Creating a fake dataset in MuJoCo for a specific task to fine-tune a VLA on. In this case picking a colored box.

In [ ]:
!python g1_walk_to_target/generate_dataset.py --mujoco-repo unitree_mujoco --episodes 500 \
    --repo-id danielpappa/g1_walk_to_target --root g1_walk_to_target/data/g1_walk_to_target

In [ ]:
from lerobot.datasets.lerobot_dataset import LeRobotDataset

LeRobotDataset("danielpappa/g1_walk_to_target", root="g1_walk_to_target/data/g1_walk_to_target").push_to_hub()

In [ ]:
dataset = LeRobotDataset("danielpappa/g1_walk_to_target")
print(dataset)

In [ ]:
# Inspecting a frame

!lerobot-dataset-viz \
    --repo-id danielpappa/g1_walk_to_target \
    --episode-index 0

### Part 1: Fine-Tuning a Vision Language Action Model

This section guides through the process of fine-tuning models available on Hugging Face using datasets from LeRobot.
This allows a generative model to learn to respond using continuous action tokens, directly transferable and interpretable by a robot's actuators, instead of with plain text responses.

Various concepts such as LoRA (peft) fine-tuning or image transformations can be explored for resource efficiency, and the final model is then tried out locally on a G1 in the MuJoCo simulator.

Note that this has been trained on TU Wien Datalab for resource constraints.

In [ ]:
!caffeinate -s !lerobot-train \
  --policy.path=lerobot/smolvla_base \
  --dataset.repo_id=danielpappa/g1_walk_to_target \
  --policy.repo_id=danielpappa/my_smolvla_walker \
  --rename_map='{"observation.images.head": "observation.images.camera1"}' \
  --policy.chunk_size=16 \
  --policy.n_action_steps=4 \
  --policy.optimizer_lr=7e-5 \
  --policy.optimizer_weight_decay=1e-4 \
  --steps=12000 \
  --policy.scheduler_warmup_steps=400 \
  --policy.scheduler_decay_steps=12000 \
  --policy.scheduler_decay_lr=5e-6 \
  --dataset.image_transforms.enable=true \
  --batch_size=32 \
  --num_workers=4 \
  --prefetch_factor=2 \
  --persistent_workers=true \
  --log_freq=200 \
  --save_freq=2000 \
  --env_eval_freq=0 \
  --policy.push_to_hub=false \
  --output_dir="outputs/train/my_smolvla_walker_${time_id}" \
  --job_name="my_smolvla_walker_training_${time_id}" \
  --policy.device=cuda \
  --trackio.enable=true

### Part 2: Running Inference

If inference is performed on MacOS, the following command may be run in the terminal and cannot be run inside a notebook, as it needs to run on the main thread:

```bash
mjpython g1_walk_to_target/rollout_policy.py --checkpoint ./outputs/train/my_smolvla_walker/checkpoints/last/pretrained_model --headless --episodes 30 --color random --seed 123
```